# Detectron2 segmentation for idtracker.ai

Takes the frames you annotated and carries a whole project the rest of the way:
train a Mask R-CNN, export contours for every clip from that setup, and write
the parameter files idtracker.ai needs.

## Before you open this

Three things happen in the Segmentation App, on your own machine, because they
want your eyes on the footage:

1. **Enhancement** - judged against a real frame, with a raw/enhanced wipe.
2. **Sample frames** - spread across every clip from the setup, in proportion
   to length.
3. **Annotate** - LabelMe, one polygon per animal. It cannot run here: Colab
   has no display.

Then the app's **Get Colab bundle...** button gives you `colab_bundle.zip`,
which contains this notebook.

This notebook is **steps 4 to 12**: build the dataset, train, export contours,
and write the parameter files for tracking. The numbering continues from the
app's, so step 4 here follows step 3 there. The two *Setup* sections before it
are not steps - they mount Drive, unpack the bundle and locate the videos, and
you run them at the start of every session.

## What to put on Drive

```
idtrackerai_detectron2/
  colab_bundle.zip      <- from the app, or idtrackerai_d2_bundle
  annotate/             <- your frames and their .json annotations
  parameters.toml       <- optional, from the app's Save parameters
```

The **videos** can live anywhere on Drive - set `VIDEOS` in the config cell to
wherever they are. They do not have to sit inside the project folder, which
matters when they are tens of GB. They are needed from section 8 onwards, when
the model is run over them.

`dataset/`, `model/`, `contours/`, `sessions/` and `wheels/` are created here.

## The long stage

**The export cannot finish in one Colab session.** At a plausible 10 fps, 48
clips of 30 000 frames is about 40 hours. So it is resumable: each clip writes
its own file, and rerunning skips whatever is already done. Reconnect and run
the export cell again as many times as it takes.

Set the runtime to a GPU first: **Runtime > Change runtime type > T4/L4/A100**.

## Setup: runtime, Drive and the bundle

In [ ]:
# Check the runtime is a GPU one before anything else. Everything below assumes it.
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or
      "No GPU. Runtime -> Change runtime type -> GPU, then rerun.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# ---- the only block you should need to edit --------------------------------
PROJECT  = Path('/content/drive/MyDrive/idtrackerai_detectron2')

# Where the videos actually live. Anywhere on Drive is fine - they do not have
# to be inside PROJECT. Point this at the real folder.
VIDEOS   = PROJECT / 'clips'
VIDEO_GLOB = '*.mp4'

BUNDLE   = PROJECT / 'colab_bundle.zip'
FRAMES   = PROJECT / 'annotate'      # sampled frames, and your annotations
DATASET  = PROJECT / 'dataset'       # train.json, val.json, images/
MODEL    = PROJECT / 'model'         # weights land here
CONTOURS = PROJECT / 'contours'      # one .h5 per clip lands here
WHEELS   = PROJECT / 'wheels'        # cached detectron2 build

CODE     = Path('/content/code')     # bundle unpacked here
CACHE    = Path('/content/cache')    # local video copies, not on Drive

SEED            = 0        # fixes the train/validation split
N_ANIMALS       = 5        # expected animals per frame
CLASS_NAME      = 'fish'   # the label you will draw with in LabelMe
VAL_FRACTION    = 0.15
SCORE_THRESHOLD = 0.7
ON_OVERLAP      = 'merge'  # 'merge' keeps crossings for idtracker.ai to resolve
EPOCHS          = 40

# The enhancement is NOT set here. It was chosen in the Segmentation App and
# written beside the frames as preprocess_profile.json, which travels into the
# dataset and then into the weights, so the model is trained and run on the
# same kind of image it was annotated on. Setting it again here could only
# introduce a disagreement.
# ----------------------------------------------------------------------------

for d in (FRAMES, MODEL, CONTOURS, WHEELS):
    d.mkdir(parents=True, exist_ok=True)

for label, path in [('bundle', BUNDLE), ('videos', VIDEOS)]:
    print(f"{label:8s} {'OK     ' if path.exists() else 'MISSING'}  {path}")

In [ ]:
# Unpack the tools. The bundle keeps tools/ and src/ side by side because the
# exporter loads write_contours from src/ when idtracker.ai is not installed.
import shutil, sys, zipfile

if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE) as zf:
    zf.extractall(CODE)

TOOLS = CODE / 'tools'
expected = [
    TOOLS / 'dataset.py',
    TOOLS / 'preprocessing.py',
    TOOLS / 'errors.py',
    TOOLS / 'train_detectron2.py',
    TOOLS / 'detectron2_export_contours.py',
    TOOLS / 'check_videos.py',
    CODE / 'src/idtrackerai/base/animals_detection/external_contours.py',
]
print('\n'.join(f"{'OK     ' if p.is_file() else 'MISSING'}  {p}" for p in expected))
if any(not p.is_file() for p in expected):
    raise SystemExit('Bundle is incomplete. Rebuild it with idtrackerai_d2_bundle.')

# The pipeline modules import each other by bare name, so a flat folder on the
# path is all they need. No idtracker.ai installation is involved.
if str(TOOLS) not in sys.path:
    sys.path.insert(0, str(TOOLS))

## Setup: find the videos

Inference reads every frame of every clip, so the videos have to be on this
runtime's Drive mount. This section fails loudly now rather than forty minutes
into a run.

A folder someone shared with you is **not reachable by path** until you add a
shortcut to it in My Drive: open *Shared with me*, right-click the folder,
*Organise → Add shortcut to Drive*. Shared drives, when enabled, appear under
`/content/drive/Shareddrives/`.

In [ ]:
# Is Drive mounted, and is the video folder actually there?
import os

mount = Path('/content/drive/MyDrive')
print(f"Drive mounted:    {mount.is_dir()}")
if not mount.is_dir():
    raise SystemExit('Drive is not mounted. Run the drive.mount cell above.')

shared = Path('/content/drive/Shareddrives')
if shared.is_dir():
    print(f"Shared drives:    {[p.name for p in shared.iterdir()][:5]}")

print(f"Video folder:     {VIDEOS}")
if not VIDEOS.is_dir():
    print('\nNOT FOUND. What is next to it:')
    parent = VIDEOS.parent
    if parent.is_dir():
        for p in sorted(parent.iterdir())[:30]:
            print(f"   {'d' if p.is_dir() else '-'} {p.name}")
    else:
        print(f'   {parent} does not exist either')
    raise SystemExit(
        'Set VIDEOS to the real folder. A "Shared with me" folder needs a '
        'shortcut added to My Drive before it has a path.')

found = sorted(VIDEOS.glob(VIDEO_GLOB))
print(f"Matching {VIDEO_GLOB}: {len(found)} file(s)")
if not found:
    others = sorted({p.suffix for p in VIDEOS.iterdir() if p.is_file()})
    print(f"  extensions present in that folder: {others}")
    raise SystemExit(f'No {VIDEO_GLOB} in {VIDEOS}. Adjust VIDEO_GLOB.')
# every later section works from this list
videos = found

for p in found[:10]:
    print(f"   {p.name}  {p.stat().st_size/1e6:.0f} MB")
if len(found) > 10:
    print(f"   ... and {len(found)-10} more")

In [ ]:
# Open every clip, decode a frame, and compare against what is already exported.
# Frame counts come from the same OpenCV call idtracker.ai uses, so a clip that
# reads as 0 frames here would also be rejected there.
!cd {TOOLS} && python check_videos.py \
    --videos {VIDEOS}/{VIDEO_GLOB} \
    --contours {CONTOURS} \
    --fps 10

In [ ]:
# Room for the local cache. Only one clip is cached at a time, but Colab's disk
# is shared with the dataset, the wheel build and the model checkpoints.
import shutil as _shutil

total, used, free = _shutil.disk_usage('/content')
biggest = max((p.stat().st_size for p in found), default=0) / 1e9
print(f"local disk: {free/1e9:.0f} GB free of {total/1e9:.0f} GB")
print(f"largest clip: {biggest:.1f} GB")
if free / 1e9 < biggest * 3:
    print("\nTight. Either drop --local-cache from the export (slower, reads"
          "\nstraight off Drive) or free space up.")
else:
    print("Enough for --local-cache.")

## 4. Your annotated frames

The frames and their `.json` annotations, together. Either put the folder on
Drive as `PROJECT/annotate/` and skip the upload cell, or upload a zip of it
here.

In [ ]:
# Upload the annotated folder, zipped. It is unpacked over FRAMES on Drive, so
# the images and your annotations end up side by side.
import zipfile
from google.colab import files

print('Choose the zip of your annotated frames.')
uploaded = files.upload()
name = next(iter(uploaded))
with zipfile.ZipFile(name) as zf:
    zf.extractall(FRAMES)

import dataset as ds
n_images = len([p for p in FRAMES.iterdir()
                if p.suffix.lower() in ('.png', '.jpg')])
n_annotated = ds.count_annotated(FRAMES)
print(f'{n_annotated} of {n_images} frames have annotations')
if n_annotated < n_images:
    print('The unannotated ones are simply left out of the dataset.')
if not n_annotated:
    raise SystemExit('No annotations found. Is this the right zip?')

## 5. Build the dataset

Every polygon is checked rather than trusted: two-point "polygons", stray
clicks, shapes off the image edge and label typos are all reported.

The train/validation split is **by recording**. Clips saved as pieces of one
recording - `trial_segment_1`, `_2`, `_3` - are kept on the same side, because
they show the same animals in the same arena minutes apart, and testing on one
after training on another measures memorisation rather than generalisation.

In [ ]:
import dataset as ds

# What the grouping will do, before it is acted on.
groups = ds.group_videos([v.stem for v in videos])
print(f'{len(videos)} clips in {len(groups)} recordings:')
for name, members in sorted(groups.items())[:10]:
    print(f'   {name:<24} {len(members)} clip(s)')
if len(groups) > 10:
    print(f'   ... and {len(groups) - 10} more')
if len(groups) < 2:
    print('\nOnly one group, so the split will fall back to per-clip.')

In [ ]:
report = ds.build_coco_dataset(ds.DatasetRequest(
    input_dir=FRAMES,
    output_dir=DATASET,
    val_fraction=VAL_FRACTION,
    single_class=CLASS_NAME,
    expected_instances=N_ANIMALS,
    seed=SEED,
))

print(f'train: {report.train_images} images, {report.train_annotations} instances')
print(f'val:   {report.val_images} images, {report.val_annotations} instances')
print(f'train recordings: {report.train_videos}')
print(f'val recordings:   {report.val_videos}')
overlap = set(report.train_videos) & set(report.val_videos)
print(f'overlap: {sorted(overlap) if overlap else "none"}')
print(f'enhancement carried forward: {report.enhancement}')

if report.problems:
    print(f'\n{len(report.problems)} problem(s):')
    for line in report.problems[:20]:
        print(f'   {line}')
if report.notes:
    print(f'\n{len(report.notes)} note(s):')
    for line in report.notes[:20]:
        print(f'   {line}')

## 6. Install Detectron2

There is no universal Detectron2 wheel — it compiles against the installed
torch/CUDA pair, which takes about ten minutes. The build is cached to Drive and
keyed by Python and torch version, so later sessions reuse it. If Colab updates
its runtime the key changes and it rebuilds once.

In [ ]:
import subprocess, sys, torch

key = f"py{sys.version_info.major}{sys.version_info.minor}-torch{torch.__version__}"
cache = WHEELS / key
cache.mkdir(parents=True, exist_ok=True)
wheels = list(cache.glob('detectron2*.whl'))
print(f"torch {torch.__version__}, CUDA {torch.version.cuda}, cache key {key}")

def run(cmd):
    print('$', ' '.join(cmd), flush=True)
    return subprocess.run(cmd).returncode

try:
    import detectron2
    print('detectron2 already importable:', detectron2.__version__)
except ImportError:
    if wheels:
        print(f'Installing cached wheel {wheels[0].name}')
        run([sys.executable, '-m', 'pip', 'install', '-q', str(wheels[0])])
    else:
        print('Building detectron2 (~10 min) and caching the wheel to Drive')
        rc = run([sys.executable, '-m', 'pip', 'wheel', '--no-deps', '-q',
                  '--wheel-dir', str(cache),
                  'git+https://github.com/facebookresearch/detectron2.git'])
        wheels = list(cache.glob('detectron2*.whl'))
        if rc or not wheels:
            raise SystemExit('detectron2 build failed; see the log above')
        run([sys.executable, '-m', 'pip', 'install', '-q', str(wheels[0])])

In [ ]:
# Verify before spending GPU time on a broken install.
import torch, detectron2
from detectron2 import model_zoo
from detectron2.config import get_cfg

print('detectron2', detectron2.__version__)
print('torch      ', torch.__version__)
print('CUDA avail ', torch.cuda.is_available())
print('device     ', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY')
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(
    'COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml'))
print('model zoo config loads OK')
assert torch.cuda.is_available(), 'No CUDA. Switch the runtime to GPU and rerun.'

## 7. Check the dataset before training

Renders a few training images with their polygons drawn on. Look at them. An
annotation that is offset, missing an animal, or attached to the wrong class
costs a whole training run to discover afterwards.

In [ ]:
!cd {TOOLS} && python train_detectron2.py \
    --dataset {DATASET} --output {MODEL} --check-dataset

In [ ]:
from IPython.display import Image, display
samples = sorted((MODEL / 'dataset_check').glob('*.png'))[:4]
print(f'{len(samples)} sample(s)')
for s in samples:
    display(Image(filename=str(s), width=600))

## 8. Train

About 40 epochs over ~540 images at batch 2 is roughly 10 800 iterations, which
takes on the order of an hour on an L4. Checkpoints land in `MODEL` every fifth
of the run, so a disconnect does not cost everything — rerun with `--resume`.

In [ ]:
!cd {TOOLS} && python train_detectron2.py \
    --dataset {DATASET} \
    --output {MODEL} \
    --epochs {EPOCHS} \
    --batch-size 2 \
    --min-size-test 640

In [ ]:
# If the session dropped mid-training, rerun this instead of the cell above.
# !cd {TOOLS} && python train_detectron2.py --dataset {DATASET} --output {MODEL} --epochs {EPOCHS} --resume

In [ ]:
import json
meta = json.loads((MODEL / 'training_metadata.json').read_text())
print(f"classes: {meta['class_names']}   images: {meta['train_images']} train / {meta['val_images']} val")
print(f"enhancement recorded: {meta['enhancement']}")
for task, metrics in meta.get('validation', {}).items():
    print(f"\n{task}:")
    for k, v in metrics.items():
        print(f"  {k:12s} {v:.3f}")
print('''
segm/AP is the one that matters for contours. Read it against how the
validation set was split: if train and val share source clips, this number is
optimistic. The dataset step groups clips by recording by default for that
reason, so the segments of one recording never straddle the split.''')

## 9. Export contours — the long stage

Resumable by design. Each clip writes `contours/<clip>.h5` and reruns skip files
that already exist, so if the session dies you reconnect and run this cell
again. Videos are copied to local disk first because decoding straight off the
Drive mount is much slower than the copy.

Run this cell as many times as it takes.

In [ ]:
!cd {TOOLS} && python detectron2_export_contours.py \
    --videos {VIDEOS}/{VIDEO_GLOB} \
    --weights {MODEL}/model_final.pth \
    --output-dir {CONTOURS} \
    --local-cache {CACHE} \
    --max-instances {N_ANIMALS} \
    --score-threshold {SCORE_THRESHOLD} \
    --on-overlap {ON_OVERLAP} \
    --progress-every 1000

In [ ]:
# How far along is the batch, and how much longer at the observed rate?
import json

clips = sorted(VIDEOS.glob(VIDEO_GLOB))
done  = sorted(CONTOURS.glob('*.h5'))
print(f'{len(done)}/{len(clips)} clips exported')

log_path = CONTOURS / 'export_log.json'
if log_path.is_file():
    log = json.loads(log_path.read_text())
    frames = sum(e['frames'] for e in log)
    seconds = sum(e['seconds'] for e in log)
    empty = sum(e['empty_frames'] for e in log)
    short = sum(e['short_frames'] for e in log)
    fps = frames / seconds if seconds else 0
    print(f'{frames:,} frames at {fps:.1f} fps overall')
    print(f'{empty:,} frames with no detection ({100*empty/max(frames,1):.2f}%)')
    print(f'{short:,} frames with fewer than {N_ANIMALS} ({100*short/max(frames,1):.2f}%)')
    remaining = [c for c in clips if not (CONTOURS / f'{c.stem}.h5').exists()]
    if remaining and fps:
        import cv2
        left = sum(int(cv2.VideoCapture(str(c)).get(cv2.CAP_PROP_FRAME_COUNT))
                   for c in remaining)
        print(f'\n{len(remaining)} clip(s) left, ~{left/fps/3600:.1f} h at this rate')
    elif not remaining:
        print('\nAll clips exported.')

print('''
A high "no detection" or "fewer than expected" rate is worth looking at before
tracking: it usually means the score threshold is too strict, or the footage is
genuinely hard in places. Those frames are left empty on purpose - idtracker.ai
reconstructs them from the whole video.''')

## 10. Check the contours before tracking on them

A quick look at what the model produced, before committing to it. Tracking
itself is sections 11 and 12.

**Track the same video files these contours were made from.** idtracker.ai
checks the sidecar's frame count and resolution against the video and refuses a
mismatch, so a re-encoded or re-exported copy will be rejected.

In [ ]:
import importlib.util, sys
spec = importlib.util.spec_from_file_location(
    'ec', CODE / 'src/idtrackerai/base/animals_detection/external_contours.py')
ec = importlib.util.module_from_spec(spec); spec.loader.exec_module(ec)

import cv2
total_mb = 0
for h5 in sorted(CONTOURS.glob('*.h5')):
    c = ec.ExternalContours(h5)
    mb = h5.stat().st_size / 1e6
    total_mb += mb
    video = VIDEOS / f'{h5.stem}{Path(VIDEO_GLOB).suffix}'
    verdict = 'no matching video found'
    if video.is_file():
        cap = cv2.VideoCapture(str(video))
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()
        verdict = 'matches video' if (n, w, h) == (c.n_frames, c.width, c.height) \
                  else f'MISMATCH: video is {n} frames {w}x{h}'
    print(f'{h5.name:28s} {c.n_frames:6d} frames  {c.width}x{c.height}  {mb:6.1f} MB  {verdict}')
    c.close()
print(f'\n{total_mb:.0f} MB total')

## 11. Build one parameter file per recording

Everything tracking needs except the contours was already decided in the
Segmentation App: how many animals, the area thresholds, the region of
interest, the tracking interval. Rather than retype it here, the notebook
**inherits the file the app saves** - the one you get from *Save parameters*,
the same file a legacy thresholding run would use.

Put it on Drive as `PROJECT/parameters.toml` (any `.toml` in the project folder
is found). Without one, a minimal file is written from `N_ANIMALS` and you get
idtracker.ai's defaults for the rest.

One file is written per recording, with its clips and their contour files in
matching order, so the segments of a trial are tracked as **one session** and
identities carry across the joins.

In [ ]:
# Read whatever the Segmentation App saved, if it is there.
import tomllib

candidates = sorted(PROJECT.glob('*.toml'))
BASE = {}
if candidates:
    CONFIG = candidates[0]
    BASE = tomllib.loads(CONFIG.read_text(encoding='utf-8'))
    print(f'Inheriting {CONFIG.name}')
    for key in ('number_of_animals', 'area_ths', 'intensity_ths', 'use_bkg',
                'tracking_intervals', 'roi_list', 'check_segmentation',
                'track_wo_identities', 'enhancement'):
        if key in BASE:
            print(f'   {key:22s} {BASE[key]}')
    if len(candidates) > 1:
        print(f'\n(ignoring {[c.name for c in candidates[1:]]})')
else:
    print('No .toml in the project folder, so starting from N_ANIMALS alone.')
    print('Save one from the Segmentation App to carry over your ROI, area')
    print('thresholds and tracking interval.')
    BASE = {'number_of_animals': N_ANIMALS}

# These are decided per recording, so anything inherited is replaced.
for key in ('video_paths', 'external_contours', 'name', 'session'):
    BASE.pop(key, None)

In [ ]:
# One parameter file per recording, clips and contours in matching order.
import math

SESSIONS = PROJECT / 'sessions'
SESSIONS.mkdir(parents=True, exist_ok=True)


def as_toml_value(value):
    """TOML is close enough to JSON for these values, with two exceptions:
    infinity is spelled `inf`, and Path is not a JSON type."""
    if isinstance(value, Path):
        return json.dumps(str(value))
    if isinstance(value, bool):
        return 'true' if value else 'false'
    if isinstance(value, float) and math.isinf(value):
        return 'inf' if value > 0 else '-inf'
    if isinstance(value, (list, tuple)):
        return '[' + ', '.join(as_toml_value(v) for v in value) + ']'
    if isinstance(value, dict):
        return '{' + ', '.join(f'{k} = {as_toml_value(v)}'
                               for k, v in value.items()) + '}'
    return json.dumps(value)


import json
groups = ds.group_videos([v.stem for v in videos])
written, skipped = [], []

for name, members in sorted(groups.items()):
    clips = [v for v in videos if v.stem in members]
    h5s = [CONTOURS / f'{c.stem}.h5' for c in clips]
    absent = [h.name for h in h5s if not h.is_file()]
    if absent:
        skipped.append((name, absent))
        continue

    params = dict(BASE)
    params['video_paths'] = [str(c) for c in clips]
    params['external_contours'] = [str(h) for h in h5s]
    params['name'] = name

    path = SESSIONS / f'{name}.toml'
    path.write_text(
        '\n'.join(f'{k} = {as_toml_value(v)}' for k, v in params.items()) + '\n',
        encoding='utf-8')
    written.append((name, len(clips)))

for name, n in written:
    print(f'   {name:<24} {n} clip(s) -> {SESSIONS / (name + ".toml")}')
if skipped:
    print(f'\n{len(skipped)} recording(s) have no contours yet, so no file was written:')
    for name, absent in skipped:
        print(f'   {name}: missing {absent[:3]}{"..." if len(absent) > 3 else ""}')
    print('Run the export cell again; it skips the clips it has already done.')
print(f'\n{len(written)} recording(s) ready to track.')

## 12. Track

Two ways, and the second is the one that has been tested.

**Here, on the runtime.** Convenient, since the videos and contours are already
on Drive. Tracking does not need the GPU for segmentation - the contours are
read from file - but the identification network will use it if there is one.

> **This has not been run on a Colab runtime.** I could not test it. The pieces
> are sound individually (`--track` skips the GUI, and idtracker.ai imports Qt
> under a guard so a display-less machine does not stop the import), but treat
> the first run as an experiment. If it fails, the local route below is
> unaffected, and the parameter files written above work there unchanged.

**At home.** Download `contours/` and `sessions/` from Drive and run the same
command on your own machine. The `.h5` files are small - tens of MB each rather
than the gigabytes an enhanced video would be - so this is a light download.
Point `video_paths` in each `.toml` at wherever the clips are locally.

In [ ]:
# Install idtracker.ai on the runtime. This pulls PyQt6 and LabelMe, which is
# heavier than it needs to be for a headless run; it is the price of one
# install command.
!pip install -q "git+https://github.com/mbel-tech/idtrackerai-detectron2.git"

In [ ]:
# UNTESTED ON COLAB. Runs each recording in turn; a failure in one does not
# stop the others, and every session folder is written beside the videos.
import os, subprocess

os.environ['QT_QPA_PLATFORM'] = 'offscreen'   # no display on a runtime

results = []
for path in sorted(SESSIONS.glob('*.toml')):
    print(f'=== {path.stem} ' + '=' * 40, flush=True)
    done = subprocess.run(['idtrackerai', '--parameters', str(path), '--track'])
    results.append((path.stem, done.returncode))
    print(f'=== {path.stem}: exit {done.returncode}\n', flush=True)

print('Summary')
for name, code_ in results:
    print(f'   {name:<24} {"ok" if code_ == 0 else f"FAILED ({code_})"}')
failed = [n for n, c in results if c]
if failed:
    print(f'\n{len(failed)} recording(s) failed: {failed}')
    print('The parameter files are unchanged, so the same run works locally.')

In [ ]:
# What to run at home instead, if you would rather not track here.
print('Download from Drive:')
print(f'   {CONTOURS}')
print(f'   {SESSIONS}')
print()
print('Then, on the machine with the videos:')
for path in sorted(SESSIONS.glob('*.toml')):
    print(f'   idtrackerai --parameters {path.name} --track')
print()
print('''Edit video_paths in each .toml to point at the clips locally, and
external_contours to the downloaded .h5 files. Or open one in the Segmentation
App: Segmentation -> External contours -> Load contours, which pairs the files
to the clips by name.

Background subtraction and the intensity thresholds grey out, because they take
no part once contours come from the model. Everything after segmentation -
crossing detection, fragmentation, identification, gap closing - runs
unchanged.''')